<a href="https://colab.research.google.com/github/querogab/curso-em-video-python/blob/main/Desafio1_API_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q gspread faker


In [ ]:
import json, gspread, random, os, time
from faker import Faker
import pandas as pd
from datetime import date, timedelta
from google.colab import files
from google.colab import userdata

# Autenticação com a conta Google: a criação de chave de service account
# foi bloqueada (iam.disableServiceAccountKeyCreation) e o limite de projetos foi atingido.
from google.colab import auth
import google.auth
auth.authenticate_user()
creds, _ = google.auth.default()
gc = gspread.authorize(creds)
print("Autenticado com a conta do Colab")

Autenticado com a conta do Colab


In [ ]:
SHEET_ID = userdata.get("GS_ID")
NOME_TABELA = "pacientes"
sh = gc.open_by_key(SHEET_ID)
ws = sh.sheet1
ws.update_title(NOME_TABELA)
print(sh.title, "|", ws.title, "|", sh.url)

Desafio_01_Posto_Saude | pacientes | https://docs.google.com/spreadsheets/d/1-M8cD8FFZL0Rf-mik3dr1ItrNbRigjjJ9fLdRXunA6I


In [ ]:
COLUNAS_DIMENSAO = ["paciente_id", "nome", "data_nascimento", "sexo", "bairro", "tipo_convenio"]
ws.update(range_name="A1", values=[COLUNAS_DIMENSAO])
print("Cabeçalho confere:", ws.row_values(1) == COLUNAS_DIMENSAO)

Cabeçalho confere: True


In [ ]:
fake = Faker("pt_BR")
Faker.seed(42)
random.seed(42)

def gerar_linha(i):
    # data de nascimento sorteada entre 1931 e 2026 (idade de 0 a 94 anos)
    nascimento = fake.date_between(start_date=date(1931, 7, 1), end_date=date(2026, 7, 1))

    return {
        "paciente_id": i,
        "nome": fake.name(),
        "data_nascimento": nascimento.isoformat(),
        "sexo": random.choices(["F", "M", "outro"], weights=[51, 47, 2])[0],
        "bairro": fake.bairro(),
        "tipo_convenio": random.choices(["SUS", "convenio", "particular"], weights=[68, 20, 12])[0],
    }

linhas = [gerar_linha(i) for i in range(1, 11)]
print(linhas[0])   # confere o primeiro paciente

valores = [[linha[c] for c in COLUNAS_DIMENSAO] for linha in linhas]
ws.append_rows(valores, value_input_option="RAW")
print(len(valores), "linhas enviadas")

{'paciente_id': 1, 'nome': 'João Guilherme da Cruz', 'data_nascimento': '1992-03-29', 'sexo': 'M', 'bairro': 'Etelvina Carneiro', 'tipo_convenio': 'SUS'}
10 linhas enviadas


In [ ]:
valores_lidos = ws.get_all_values()
print("Linhas (cabeçalho + dados):", len(valores_lidos))   # esperado: 11
print("Cabeçalho confere:", valores_lidos[0] == COLUNAS_DIMENSAO)
print(valores_lidos[:3])

Linhas (cabeçalho + dados): 11
Cabeçalho confere: True
[['paciente_id', 'nome', 'data_nascimento', 'sexo', 'bairro', 'tipo_convenio'], ['1', 'João Guilherme da Cruz', '1992-03-29', 'M', 'Etelvina Carneiro', 'SUS'], ['2', 'Yago Borges', '1944-10-01', 'F', 'Tupi B', 'SUS']]


In [ ]:
NOME_TABELA_FATO = "atendimentos"
COLUNAS_FATO = ["atendimento_id", "paciente_id", "data_hora_chegada", "sintoma_principal",
                "nivel_triagem", "tempo_espera_min", "profissional_responsavel", "desfecho"]
COLUNA_FK = "paciente_id"
assert COLUNA_FK in COLUNAS_FATO, "COLUNA_FK precisa ser um dos nomes de COLUNAS_FATO"
print(NOME_TABELA_FATO, "|", COLUNAS_FATO)

atendimentos | ['atendimento_id', 'paciente_id', 'data_hora_chegada', 'sintoma_principal', 'nivel_triagem', 'tempo_espera_min', 'profissional_responsavel', 'desfecho']


In [ ]:
SINTOMAS = ["dor no peito", "febre", "corte/ferimento", "falta de ar", "dor abdominal",
            "tontura", "dor de cabeça", "vômito/náusea", "alergia", "check-up de rotina"]
# Média de espera (minutos) por nível de triagem, conforme a documentação do tema
MEDIA_ESPERA = {"vermelho": 0, "laranja": 8, "amarelo": 35, "verde": 70, "azul": 150}

def gerar_linha_fato(i, ids_dim):
    # Valor de origem: o nível de triagem define a espera e o desfecho
    nivel = random.choices(["vermelho", "laranja", "amarelo", "verde", "azul"],
                           weights=[2, 8, 35, 40, 15])[0]

    # Espera: distribuição normal em torno da média do nível, desvio de 35% da média + 5, sem negativo
    media = MEDIA_ESPERA[nivel]
    espera = max(0, round(random.gauss(media, media * 0.35 + 5)))

    # Casos vermelho e laranja nunca são "nao_compareceu"
    if nivel in ["vermelho", "laranja"]:
        desfecho = random.choices(["alta", "encaminhado", "internado"], weights=[72, 16, 5])[0]
    else:
        desfecho = random.choices(["alta", "encaminhado", "nao_compareceu", "internado"],
                                  weights=[72, 16, 6, 5])[0]

    cargo = random.choice(["Enfermeiro", "Enfermeira", "Tecnico Enfermagem"])

    return {
        "atendimento_id": i,
        "paciente_id": random.choice(ids_dim),   # chave estrangeira: só IDs que existem em pacientes
        "data_hora_chegada": fake.date_time_between(start_date=date(2025, 1, 1),
                                                    end_date=date(2026, 8, 1)).strftime("%Y-%m-%d %H:%M:%S"),
        "sintoma_principal": random.choice(SINTOMAS),
        "nivel_triagem": nivel,
        "tempo_espera_min": espera,
        "profissional_responsavel": f"{cargo} {random.randint(1, 24):02d}",
        "desfecho": desfecho,
    }

print(gerar_linha_fato(1, [1, 2, 3]))

{'atendimento_id': 1, 'paciente_id': 1, 'data_hora_chegada': '2026-04-24 13:35:44', 'sintoma_principal': 'febre', 'nivel_triagem': 'verde', 'tempo_espera_min': 61, 'profissional_responsavel': 'Enfermeira 13', 'desfecho': 'alta'}


In [ ]:
N_DIM = 10_500
N_FATO = 42_000

Faker.seed(42)
random.seed(42)

tabela_dim = pd.DataFrame([gerar_linha(i) for i in range(1, N_DIM + 1)], columns=COLUNAS_DIMENSAO)

ids_dim = list(tabela_dim[COLUNAS_DIMENSAO[0]])
tabela_fato = pd.DataFrame([gerar_linha_fato(i, ids_dim) for i in range(1, N_FATO + 1)], columns=COLUNAS_FATO)

BASES_GERADAS = {NOME_TABELA: tabela_dim, NOME_TABELA_FATO: tabela_fato}
print({nome: len(df) for nome, df in BASES_GERADAS.items()}, "| total:", len(tabela_dim) + len(tabela_fato))

{'pacientes': 10500, 'atendimentos': 42000} | total: 52500


In [ ]:
assert list(tabela_dim.columns) == COLUNAS_DIMENSAO, "Colunas diferentes do esquema na tabela dimensão"
assert list(tabela_fato.columns) == COLUNAS_FATO, "Colunas diferentes do esquema na tabela fato"
assert tabela_dim[COLUNAS_DIMENSAO[0]].is_unique, "ID repetido na tabela dimensão"
assert tabela_fato[COLUNA_FK].isin(tabela_dim[COLUNAS_DIMENSAO[0]]).all(), "Chave estrangeira sem ID na dimensão"
assert len(tabela_dim) + len(tabela_fato) >= 20_000, "Abaixo do mínimo (20.000 linhas somadas)."

os.makedirs("bases_geradas", exist_ok=True)
for nome, df in BASES_GERADAS.items():
    caminho = f"bases_geradas/{nome}.csv"
    df.to_csv(caminho, index=False, encoding="utf-8")
    lido = pd.read_csv(caminho)
    assert len(lido) == len(df) and list(lido.columns) == list(df.columns), f"CSV diferente em {nome}"
    print(nome + ".csv ->", len(lido), "linhas")
    files.download(caminho)

pacientes.csv -> 10500 linhas


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

atendimentos.csv -> 42000 linhas


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
def enviar(sh, df, nome_aba, lote=5000, pausa=2):
    try:
        aba = sh.worksheet(nome_aba)
        aba.clear()
    except gspread.WorksheetNotFound:
        aba = sh.add_worksheet(title=nome_aba, rows=len(df) + 1, cols=len(df.columns))
    aba.resize(rows=len(df) + 1, cols=len(df.columns))
    aba.update(range_name="A1", values=[list(df.columns)])
    valores = df.astype(object).where(df.notna(), "").values.tolist()
    for i in range(0, len(valores), lote):
        aba.append_rows(valores[i:i + lote], value_input_option="RAW")
        print(f"  {nome_aba}: {min(i + lote, len(valores))} / {len(valores)}")
        time.sleep(pausa)
    lidas = len(aba.col_values(1)) - 1
    if lidas != len(df):
        raise RuntimeError(f"Aba {nome_aba}: enviei {len(df)}, li {lidas}.")
    print(f"OK {nome_aba}: {len(df)} linhas na planilha")

In [ ]:
for nome, df in BASES_GERADAS.items():
    enviar(sh, df, nome)

  pacientes: 5000 / 10500
  pacientes: 10000 / 10500
  pacientes: 10500 / 10500
OK pacientes: 10500 linhas na planilha
  atendimentos: 5000 / 42000
  atendimentos: 10000 / 42000
  atendimentos: 15000 / 42000
  atendimentos: 20000 / 42000
  atendimentos: 25000 / 42000
  atendimentos: 30000 / 42000
  atendimentos: 35000 / 42000
  atendimentos: 40000 / 42000
  atendimentos: 42000 / 42000
OK atendimentos: 42000 linhas na planilha


In [ ]:
total = sum(len(df) for df in BASES_GERADAS.values())
print("Total de linhas:", total)
assert total >= 20_000, "Abaixo do mínimo (20.000 linhas somadas)."

for nome, df in BASES_GERADAS.items():
    w = sh.worksheet(nome)
    assert w.row_values(1) == list(df.columns), f"Cabeçalho diferente em {nome}"
    assert len(w.col_values(1)) - 1 == len(df), f"Contagem diferente em {nome}"

fk_fato = BASES_GERADAS[NOME_TABELA_FATO][COLUNA_FK]
ids_dim = BASES_GERADAS[NOME_TABELA][COLUNAS_DIMENSAO[0]]
assert fk_fato.isin(ids_dim).all(), "Há chave estrangeira sem ID correspondente na tabela dimensão"
print("A planilha confere com a base gerada.")

Total de linhas: 52500
A planilha confere com a base gerada.
